In [9]:
# custom
from dataset import create_dataset
from model import Heatmap2D_Model
from transforms import train_transform, test_transform
from config import cfg

# torch
import torch
from torch.utils.data import DataLoader
#
from torch.optim.adamw import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.nn import MSELoss

# extra
from tqdm import tqdm
import os

## Configurations

In [10]:
root_path = "/home/furkan/projects/mebar-orthognatic"
manifest_path = "/home/furkan/projects/mebar-orthognatic/data/processed/manifest.csv"

# from config
epochs = cfg.epochs
checkpoint_freq = cfg.checkpoint_freq

# train
lr = 0.001
loss_obj = MSELoss
optimizer_obj = AdamW
lr_scheduler_obj = CosineAnnealingLR

## Preparetions

In [11]:
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    # Tiny GPU tensor allocation test
    x = torch.ones(1, device="cuda")
    print(f"GPU Test Tensor: {x}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CUDA Available: True
Device: NVIDIA GeForce RTX 3060 Laptop GPU
GPU Test Tensor: tensor([1.], device='cuda:0')


In [12]:
train_dataset, test_dataset = create_dataset(
    manifest_path,
    root_path, 
    train_size=0.8, 
    train_transform=train_transform,
    test_transform=test_transform
    )

In [13]:
print(len(train_dataset))
print(len(test_dataset))

153
39


In [14]:
def sample_collate_fn(batch):
    return {
        "name": [item.name for item in batch],
        "image": torch.stack([item.image for item in batch], dim=0),
        "points": torch.stack([item.points for item in batch], dim=0),
        "heatmaps": torch.stack([item.heatmaps for item in batch], dim=0)
    }

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=sample_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=sample_collate_fn
)

In [15]:
model = Heatmap2D_Model()
optimizer = optimizer_obj(model.parameters(), lr=lr, weight_decay=0.01)
lr_scheduler = lr_scheduler_obj(optimizer, T_max=epochs, eta_min=1e-6)

## Train Loop

In [16]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

checkpoint_dir = os.path.join(root_path, 'ai_models/heatmap2D/outputs/checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

model = model.to(device)
loss_fn = loss_obj().to(device)

best_path = None
best_val_loss = float('inf')
history = []

for epoch in range(1, epochs + 1):
    # 1. Training Phase
    model.train()
    train_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
    for batch in pbar:
        # Handle dict/dataclass access safely
        images = (batch['image'] if isinstance(batch, dict) else batch.image).to(device, non_blocking=True)
        heatmaps = (batch['heatmaps'] if isinstance(batch, dict) else batch.heatmaps).to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        preds = model(images)
        loss = loss_fn(preds, heatmaps)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    avg_train_loss = train_loss / len(train_loader)
    
    # 2. Validation Phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        pbar_val = tqdm(test_loader, desc=f"Epoch {epoch}/{epochs} [Val]")
        for batch in pbar_val:
            images = (batch['image'] if isinstance(batch, dict) else batch.image).to(device, non_blocking=True)
            heatmaps = (batch['heatmaps'] if isinstance(batch, dict) else batch.heatmaps).to(device, non_blocking=True)
            
            preds = model(images)
            loss = loss_fn(preds, heatmaps)
            
            val_loss += loss.item()
            pbar_val.set_postfix({"loss": f"{loss.item():.4f}"})
            
    avg_val_loss = val_loss / len(test_loader)
    current_lr = lr_scheduler.get_last_lr()[0] if hasattr(lr_scheduler, "get_last_lr") else optimizer.param_groups[0]['lr']
    
    # Update lr_scheduler
    if isinstance(lr_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
        lr_scheduler.step(avg_val_loss)
    else:
        lr_scheduler.step()
    
    # 3. Log epoch metrics to history
    epoch_metrics = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "lr": current_lr
    }
    history.append(epoch_metrics)
    
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | LR: {current_lr:.6f}")
    
    # 4. Save Best Model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        if best_path:
            os.remove(best_path)
        best_path = os.path.join(checkpoint_dir, f"best_ep{epoch}.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': lr_scheduler.state_dict(),
            'val_loss': avg_val_loss,
        }, best_path)
        print(f"⭐ New best model saved with Val Loss: {avg_val_loss:.5f}")
    
    # 5. Periodic Checkpoint
    if epoch % checkpoint_freq == 0:
        checkpoint_path = os.path.join(checkpoint_dir, f"model_epoch_{epoch}.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': lr_scheduler.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
        }, checkpoint_path)

# --- Post-Training Artifacts & Report ---

# Save training history to CSV & JSON
df_history = pd.DataFrame(history)
df_history.to_csv(os.path.join(checkpoint_dir, "training_history.csv"), index=False)

# Save training curve plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(df_history['epoch'], df_history['train_loss'], label='Train Loss')
plt.plot(df_history['epoch'], df_history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(df_history['epoch'], df_history['lr'], label='Learning Rate', color='orange')
plt.xlabel('Epoch')
plt.ylabel('LR')
plt.title('Learning Rate Schedule')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, "training_curves.png"), dpi=300)
plt.close()

print("\n Training complete. Metrics and visual curves saved successfully.")

Epoch 1/500 [Val]: 100%|██████████| 5/5 [00:01<00:00,  3.17it/s, loss=0.0268]


Epoch 1 | Train Loss: 0.06095 | Val Loss: 0.02776 | LR: 0.001000
⭐ New best model saved with Val Loss: 0.02776


Epoch 2/500 [Val]: 100%|██████████| 5/5 [00:01<00:00,  2.74it/s, loss=0.0208]


Epoch 2 | Train Loss: 0.01987 | Val Loss: 0.01776 | LR: 0.001000
⭐ New best model saved with Val Loss: 0.01776


Epoch 3/500 [Val]: 100%|██████████| 5/5 [00:01<00:00,  3.01it/s, loss=0.0083]


...

Epoch 498 | Train Loss: 0.00003 | Val Loss: 0.00008 | LR: 0.000001


Epoch 499/500 [Val]: 100%|██████████| 5/5 [00:01<00:00,  3.07it/s, loss=0.0001]


Epoch 499 | Train Loss: 0.00002 | Val Loss: 0.00008 | LR: 0.000001


Epoch 500/500 [Val]: 100%|██████████| 5/5 [00:01<00:00,  3.69it/s, loss=0.0001]


Epoch 500 | Train Loss: 0.00003 | Val Loss: 0.00008 | LR: 0.000001

 Training complete. Metrics and visual curves saved successfully.
